In [2]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 53.7 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import re
import string

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from gensim.models import Word2Vec

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [7]:
url = "https://raw.githubusercontent.com/clairett/pytorch-sentiment-classification/master/data/SST2/train.tsv"

df = pd.read_csv(
    url,
    sep='\t',
    header=None,
    names=['sentence', 'label']
)
df.head()

,sentence,label
0,"a stirring , funny and finally transporting re...",1
1,apparently reassembled from the cutting room f...,0
2,they presume their audience wo n't sit still f...,0
3,this is a visually stunning rumination on love...,1
4,jonathan parker 's bartleby should have been t...,1


In [8]:
print("Before Cleaning:")
print(df['sentence'][0])

Before Cleaning:
a stirring , funny and finally transporting re imagining of beauty and the beast and 1930s horror films


In [10]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):

    # Lowercase
    text = text.lower()

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Tokenization
    words = text.split()

    # Remove stopwords and lemmatize
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]

    return " ".join(words)

In [11]:
df['cleaned_sentence'] = df['sentence'].apply(clean_text)

df[['sentence', 'cleaned_sentence']].head()

,sentence,cleaned_sentence
0,"a stirring , funny and finally transporting re...",stirring funny finally transporting imagining ...
1,apparently reassembled from the cutting room f...,apparently reassembled cutting room floor give...
2,they presume their audience wo n't sit still f...,presume audience wo nt sit still sociology les...
3,this is a visually stunning rumination on love...,visually stunning rumination love memory histo...
4,jonathan parker 's bartleby should have been t...,jonathan parker bartleby end modern office ano...


In [12]:
print("After Cleaning:")
print(df['cleaned_sentence'][0])

After Cleaning:
stirring funny finally transporting imagining beauty beast horror film


In [13]:
tokenized_sentences = [sentence.split() for sentence in df['cleaned_sentence']]

In [39]:
w2v_model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=200,
    window=3,
    min_count=1,
    workers=4,
    sg=1,
    epochs=50
)

In [40]:
words = ['good', 'bad', 'movie']

for word in words:
    if word in w2v_model.wv:
        print(f"\nEmbedding for '{word}':")
        print(w2v_model.wv[word][:10])


Embedding for 'good':
[-0.4038851  -0.11534014  0.24774359  0.09860883 -0.00802957 -0.7768554
  0.23928666  0.49123058 -0.40954056 -0.29548302]

Embedding for 'bad':
[ 0.05591904 -0.7146858  -0.30247492  0.19625433 -1.1617503  -0.2642934
  0.6612194   0.9686368  -0.970968    0.6299864 ]

Embedding for 'movie':
[-0.20734835  0.14029719  0.24655406  0.07924827  0.4382293   0.04914895
 -0.05516614  0.4309086  -0.6525542   0.47530636]


In [42]:
print("Similarity between movie and film:",
      w2v_model.wv.similarity('movie', 'film'))

print("Similarity between story and plot:",
      w2v_model.wv.similarity('story', 'plot'))

print("Similarity between bad and awful:",
      w2v_model.wv.similarity('bad', 'awful'))

Similarity between movie and film: 0.21939914
Similarity between story and plot: 0.22569713
Similarity between bad and awful: 0.4059257


In [17]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(df['cleaned_sentence'])

X = tokenizer.texts_to_sequences(df['cleaned_sentence'])

In [19]:
max_length = 100

X = pad_sequences(X, maxlen=max_length)

y = df['label']

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [21]:
model = Sequential()

model.add(Embedding(input_dim=5000, output_dim=64, input_length=max_length))

model.add(LSTM(64))

model.add(Dense(1, activation='sigmoid'))

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [22]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
70/70 ━━━━━━━━━━━━━━━━━━━━ 11s 114ms/step - accuracy: 0.5707 - loss: 0.6795 - val_accuracy: 0.5758 - val_loss: 0.6567
Epoch 2/5
70/70 ━━━━━━━━━━━━━━━━━━━━ 10s 112ms/step - accuracy: 0.7346 - loss: 0.5363 - val_accuracy: 0.7356 - val_loss: 0.5509
Epoch 3/5
70/70 ━━━━━━━━━━━━━━━━━━━━ 9s 127ms/step - accuracy: 0.8638 - loss: 0.3318 - val_accuracy: 0.7708 - val_loss: 0.5097
Epoch 4/5
70/70 ━━━━━━━━━━━━━━━━━━━━ 14s 173ms/step - accuracy: 0.9223 - loss: 0.1967 - val_accuracy: 0.7581 - val_loss: 0.6010
Epoch 5/5
70/70 ━━━━━━━━━━━━━━━━━━━━ 14s 78ms/step - accuracy: 0.9598 - loss: 0.1237 - val_accuracy: 0.7581 - val_loss: 0.7483


In [23]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)

44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.7753 - loss: 0.6711
Test Accuracy: 0.775288999080658


In [44]:
y_pred = (model.predict(X_test) > 0.5).astype("int32")

print(classification_report(y_test, y_pred))

44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step
              precision    recall  f1-score   support

           0       0.79      0.74      0.76       671
           1       0.77      0.81      0.79       713

    accuracy                           0.78      1384
   macro avg       0.78      0.77      0.77      1384
weighted avg       0.78      0.78      0.77      1384



In [31]:
improved_model = Sequential()

improved_model.add(
    Embedding(input_dim=5000, output_dim=128, input_length=max_length)
)

improved_model.add(LSTM(64, dropout=0.2))

improved_model.add(Dense(1, activation='sigmoid'))

improved_model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

improved_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [32]:
history2 = improved_model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/10
70/70 ━━━━━━━━━━━━━━━━━━━━ 13s 145ms/step - accuracy: 0.5989 - loss: 0.6668 - val_accuracy: 0.6724 - val_loss: 0.6075
Epoch 2/10
70/70 ━━━━━━━━━━━━━━━━━━━━ 8s 119ms/step - accuracy: 0.8238 - loss: 0.4193 - val_accuracy: 0.7581 - val_loss: 0.4950
Epoch 3/10
70/70 ━━━━━━━━━━━━━━━━━━━━ 10s 150ms/step - accuracy: 0.9074 - loss: 0.2385 - val_accuracy: 0.7590 - val_loss: 0.5518
Epoch 4/10
70/70 ━━━━━━━━━━━━━━━━━━━━ 10s 137ms/step - accuracy: 0.9460 - loss: 0.1453 - val_accuracy: 0.7635 - val_loss: 0.7070
Epoch 5/10
70/70 ━━━━━━━━━━━━━━━━━━━━ 9s 132ms/step - accuracy: 0.9684 - loss: 0.0973 - val_accuracy: 0.7617 - val_loss: 0.8400
Epoch 6/10
70/70 ━━━━━━━━━━━━━━━━━━━━ 8s 119ms/step - accuracy: 0.9794 - loss: 0.0695 - val_accuracy: 0.7545 - val_loss: 0.9106
Epoch 7/10
70/70 ━━━━━━━━━━━━━━━━━━━━ 10s 136ms/step - accuracy: 0.9810 - loss: 0.0541 - val_accuracy: 0.7545 - val_loss: 1.0694
Epoch 8/10
70/70 ━━━━━━━━━━━━━━━━━━━━ 9s 136ms/step - accuracy: 0.9874 - loss: 0.0356 - val_accuracy

In [33]:
loss2, accuracy2 = improved_model.evaluate(X_test, y_test)

print("Improved Model Accuracy:", accuracy2)

44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.7579 - loss: 1.2981
Improved Model Accuracy: 0.7579479813575745


In [45]:
y_pred2 = (improved_model.predict(X_test) > 0.5).astype("int32")

print(classification_report(y_test, y_pred2))

44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step
              precision    recall  f1-score   support

           0       0.78      0.70      0.74       671
           1       0.74      0.81      0.77       713

    accuracy                           0.76      1384
   macro avg       0.76      0.76      0.76      1384
weighted avg       0.76      0.76      0.76      1384



In [35]:
print("Original Model Accuracy:", accuracy)
print("Improved Model Accuracy:", accuracy2)

Original Model Accuracy: 0.775288999080658
Improved Model Accuracy: 0.7579479813575745


Although the improved model used a larger embedding dimension and dropout regularization, the performance slightly decreased. This may be due to overfitting prevention caused by dropout or the limited size of the SST2 dataset.

Part 2

q1

### Difference Between Soft Attention and Hard Attention

Soft Attention assigns different weights to all input elements and is fully differentiable, which makes it trainable using backpropagation.

Hard Attention selects only specific parts of the input and ignores the rest. It is usually non-differentiable and harder to train.

Q2

### Bahdanau vs Luong Attention

Bahdanau Attention calculates attention before the decoder generates the output and uses a feedforward neural network for alignment scoring.


Luong Attention calculates attention after generating the decoder hidden state and uses simpler dot-product scoring, making it computationally faster.

Part 3

### Importance of Attention Mechanisms

Attention mechanisms help sequence models focus on the most important parts of the input sequence. They improve performance in tasks such as machine translation, text summarization, and sentiment analysis by handling long-range dependencies more effectively.